# Custom modalities and likelihoods

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashford-A/UniVI/blob/main/docs/tutorials/custom_modalities.ipynb)

UniVI is not limited to RNA + ATAC or RNA + protein. Any number of modalities can be combined, each with its own encoder, decoder and likelihood. This notebook shows how to wire up new assays using small **synthetic** data (so it runs anywhere in about a minute), covering:

- three modalities at once
- a **negative binomial** likelihood on raw counts
- a **beta-binomial** likelihood on methylation-style successes and coverage, which needs extra reconstruction targets
- a **learned gating** network that weights modalities per cell
- an (experimental) **transformer encoder** for one modality

The numbers produced here only demonstrate the mechanics. For real tri-modal analyses see the TEA-seq and scNMT-seq sections of the [paper reproduction](../reproducibility/index.md) pages.

In [1]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "univi[tutorials]>=1.0"

In [2]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

from univi import ModalityConfig, TrainingConfig, UniVIConfig, UniVIMultiModalVAE, UniVITrainer
from univi.config import TokenizerConfig, TransformerConfig
from univi.evaluation import (compute_foscttm, cross_modal_predict, encode_adata, encode_fused_adata_pair,
                              encode_moe_gates_from_tensors)
from univi.utils.seed import set_seed
from univi.workflows import make_loader

set_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

C:\Users\ashfo\micromamba\envs\univi-release-050\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
N_EPOCHS = 60
N_CELLS = 2000

## Synthetic tri-modal data

Three cell states drive all modalities: RNA counts (300 genes), protein counts (20 antibodies), and a methylation-like assay where each of 150 regions has a number of methylated reads (successes) out of a variable number of covering reads (coverage).

In [4]:
rng = np.random.default_rng(0)
state = rng.integers(0, 3, N_CELLS)
obs = pd.DataFrame({"state": pd.Categorical(state.astype(str))}, index=[f"cell{i}" for i in range(N_CELLS)])

def programs(n_features, scale):
    return rng.normal(0, scale, (3, n_features))[state]

rna_counts = rng.poisson(np.exp(0.5 + programs(300, 1.0))).astype(np.float32)
adt_counts = rng.poisson(np.exp(3.0 + programs(20, 0.8))).astype(np.float32)
coverage = rng.poisson(8, (N_CELLS, 150)).astype(np.float32)
p_meth = 1 / (1 + np.exp(-programs(150, 1.5)))
successes = rng.binomial(coverage.astype(int), p_meth).astype(np.float32)

rna = ad.AnnData(sp.csr_matrix(rna_counts), obs=obs.copy())
adt = ad.AnnData(adt_counts, obs=obs.copy())
clr = np.log1p(adt_counts)
adt.X = (clr - clr.mean(1, keepdims=True)).astype(np.float32)        # CLR for a Gaussian likelihood

meth = ad.AnnData(np.divide(successes, coverage, out=np.full_like(successes, 0.5), where=coverage > 0),
                  obs=obs.copy())                                    # .X: methylated fraction (encoder input)
meth.layers["successes"], meth.layers["coverage"] = successes, coverage

train_idx, val_idx = np.arange(0, int(0.9 * N_CELLS)), np.arange(int(0.9 * N_CELLS), N_CELLS)
subset = lambda d, idx: {k: v[idx].copy() for k, v in d.items()}
data = {"rna": rna, "adt": adt, "meth": meth}

## Choosing a likelihood

The decoder likelihood should match what the modality's `.X` (or reconstruction targets) contains:

| data in the model input | `likelihood` |
| --- | --- |
| normalized / log / z-scored / CLR / LSI values | `"gaussian"` |
| raw counts | `"nb"`, `"zinb"`, `"poisson"` |
| binary (e.g. binarized peaks) | `"bernoulli"` |
| proportions in (0, 1) | `"beta"` |
| successes out of trials (methylation, allele counts) | `"binomial"`, `"beta_binomial"` + reconstruction targets |
| integer class codes | `"categorical"` |

The encoder receives the same `.X`; count models do not log-transform inputs internally. Gaussian likelihoods on normalized inputs gave the best cross-modal alignment in the paper's integration benchmarks, while count likelihoods are useful when you want decoders that output counts.

Beta-binomial modalities read their successes and trials from layers named in `recon_targets_spec`:

In [5]:
recon_targets = {"meth": {"successes_layer": "successes", "total_count_layer": "coverage"}}

cfg = UniVIConfig(
    latent_dim=8, beta=1.0, gamma=2.0,
    modalities=[
        ModalityConfig("rna", 300, [128, 64], [64, 128], likelihood="nb"),
        ModalityConfig("adt", 20, [32], [32], likelihood="gaussian", recon_weight=2.0),
        ModalityConfig("meth", 150, [64], [64], likelihood="beta_binomial"),
    ],
)
model = UniVIMultiModalVAE(cfg, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
UniVITrainer(model,
             make_loader(subset(data, train_idx), batch_size=128, shuffle=True, drop_last=True,
                         recon_targets_spec=recon_targets),
             make_loader(subset(data, val_idx), batch_size=512, recon_targets_spec=recon_targets),
             TrainingConfig(n_epochs=N_EPOCHS, lr=1e-3, device=device, log_every=20)).fit();

[2026-09-21 01:04:17,892] [UniVITrainer] [INFO] TrainingConfig:


[2026-09-21 01:04:17,893] [UniVITrainer] [INFO]   n_epochs: 60


[2026-09-21 01:04:17,893] [UniVITrainer] [INFO]   batch_size: 256


[2026-09-21 01:04:17,893] [UniVITrainer] [INFO]   lr: 0.001


[2026-09-21 01:04:17,894] [UniVITrainer] [INFO]   weight_decay: 0.0


...

Training UniVI:  98%|█████████▊| 59/60 [01:14<00:01,  1.23s/it, beta=1.000, gamma=2.000, train_loss=254.3079, val_loss=253.5034]

[2026-09-21 01:05:32,885] [UniVITrainer] [INFO] [Epoch 060] New best val loss: 253.5034


Training UniVI: 100%|██████████| 60/60 [01:14<00:00,  1.23s/it, beta=1.000, gamma=2.000, train_loss=254.3079, val_loss=253.5034]

Training UniVI: 100%|██████████| 60/60 [01:14<00:00,  1.25s/it, beta=1.000, gamma=2.000, train_loss=254.3079, val_loss=253.5034]


[2026-09-21 01:05:32,917] [UniVITrainer] [INFO] Restored best model from epoch 60 (val loss=253.5034)


`recon_weight` rescales a modality's reconstruction term, which helps when modalities differ greatly in size (here protein has far fewer features than RNA).

Every encoder places cells in the same space, so any pair of modalities can be compared or translated:

In [6]:
val_data = subset(data, val_idx)
z = {m: encode_adata(model, a, modality=m, device=device, latent="modality_mean") for m, a in val_data.items()}
print({f"FOSCTTM {a}-{b}": round(compute_foscttm(z[a], z[b]), 3) for a, b in [("rna", "adt"), ("rna", "meth"), ("adt", "meth")]})

meth_from_rna = cross_modal_predict(model, val_data["rna"], src_mod="rna", tgt_mod="meth", device=device)
print("predicted methylated fraction, first cell:", np.round(meth_from_rna[0, :5], 3))

{'FOSCTTM rna-adt': 0.165, 'FOSCTTM rna-meth': 0.165, 'FOSCTTM adt-meth': 0.175}
predicted methylated fraction, first cell: [0.753 0.107 0.594 0.517 0.522]


For count and beta-binomial decoders, `cross_modal_predict` returns the decoder mean (expected counts or expected fractions).

## Learned per-cell modality weights

By default the fused posterior weights modalities by their posterior precision. A learned gating network can re-weight them per cell. It is trained only if the fused posterior enters the loss, so pair `use_moe_gating=True` with `v1_recon="moe"` (or `loss_mode="v2"`).

In [7]:
gated_cfg = UniVIConfig(latent_dim=8, beta=1.0, gamma=2.0, modalities=cfg.modalities,
                        use_moe_gating=True, moe_gating_hidden=[32])
gated = UniVIMultiModalVAE(gated_cfg, loss_mode="v1", v1_recon="moe", normalize_v1_terms=True)
UniVITrainer(gated,
             make_loader(subset(data, train_idx), batch_size=128, shuffle=True, drop_last=True,
                         recon_targets_spec=recon_targets),
             None, TrainingConfig(n_epochs=N_EPOCHS, lr=1e-3, device=device, log_every=20)).fit();

x = {m: a.X for m, a in val_data.items()}
router = encode_moe_gates_from_tensors(gated, x, device=device, kind="router_x_precision")
precision = encode_moe_gates_from_tensors(gated, x, device=device, kind="effective_precision")
pd.DataFrame({"router x precision": router["per_modality_mean"],
              "precision only": precision["per_modality_mean"]}).round(3)

[2026-09-21 01:05:33,005] [UniVITrainer] [INFO] TrainingConfig:


[2026-09-21 01:05:33,006] [UniVITrainer] [INFO]   n_epochs: 60


[2026-09-21 01:05:33,006] [UniVITrainer] [INFO]   batch_size: 256


[2026-09-21 01:05:33,006] [UniVITrainer] [INFO]   lr: 0.001


[2026-09-21 01:05:33,007] [UniVITrainer] [INFO]   weight_decay: 0.0


...

[2026-09-21 01:06:25,051] [UniVITrainer] [INFO] [Epoch 060] Train loss=253.8308 (beta=1.000, gamma=2.000)


Training UniVI:  98%|█████████▊| 59/60 [00:52<00:00,  1.15it/s, beta=1.000, gamma=2.000, train_loss=253.8308]

Training UniVI: 100%|██████████| 60/60 [00:52<00:00,  1.16it/s, beta=1.000, gamma=2.000, train_loss=253.8308]

Training UniVI: 100%|██████████| 60/60 [00:52<00:00,  1.15it/s, beta=1.000, gamma=2.000, train_loss=253.8308]

,router x precision,precision only
rna,0.665,0.372
adt,0.000,0.296
meth,0.335,0.332


## A transformer encoder (experimental)

Any modality can use a transformer encoder instead of an MLP. The tokenizer turns each cell into a set of tokens (here the 64 highest-valued features, each carrying its value, rank and a dropout indicator).

In [8]:
tf_cfg = UniVIConfig(
    latent_dim=8, beta=1.0, gamma=2.0,
    modalities=[
        ModalityConfig("rna", 300, [128, 64], [64, 128], likelihood="nb", encoder_type="transformer",
                       tokenizer=TokenizerConfig(mode="topk_channels", n_tokens=64,
                                                 channels=("value", "rank", "dropout")),
                       transformer=TransformerConfig(d_model=64, num_heads=4, num_layers=2, dim_feedforward=128)),
        ModalityConfig("adt", 20, [32], [32], likelihood="gaussian"),
    ],
)
tf_model = UniVIMultiModalVAE(tf_cfg, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
two = {"rna": rna, "adt": adt}
UniVITrainer(tf_model, make_loader(subset(two, train_idx), batch_size=128, shuffle=True, drop_last=True),
             make_loader(subset(two, val_idx), batch_size=512),
             TrainingConfig(n_epochs=max(1, N_EPOCHS // 3), lr=1e-3, device=device, log_every=10)).fit();
encode_adata(tf_model, val_data["rna"], modality="rna", device=device).shape

[2026-09-21 01:06:25,126] [UniVITrainer] [INFO] TrainingConfig:


[2026-09-21 01:06:25,126] [UniVITrainer] [INFO]   n_epochs: 20


[2026-09-21 01:06:25,127] [UniVITrainer] [INFO]   batch_size: 256


[2026-09-21 01:06:25,127] [UniVITrainer] [INFO]   lr: 0.001


[2026-09-21 01:06:25,127] [UniVITrainer] [INFO]   weight_decay: 0.0


...

[2026-09-21 01:06:38,905] [UniVITrainer] [INFO] [Epoch 020] New best val loss: 294.4854


Training UniVI: 100%|██████████| 20/20 [00:13<00:00,  1.43it/s, beta=1.000, gamma=2.000, train_loss=296.0893, val_loss=294.4854]

Training UniVI: 100%|██████████| 20/20 [00:13<00:00,  1.45it/s, beta=1.000, gamma=2.000, train_loss=296.0893, val_loss=294.4854]


[2026-09-21 01:06:38,938] [UniVITrainer] [INFO] Restored best model from epoch 20 (val loss=294.4854)


(200, 8)

## Checklist for a new assay

1. One AnnData per modality; identical, identically ordered `obs_names` across paired modalities.
2. Put the model input in `.X` (or an `.obsm` key passed as `X_key`) and keep raw data in a layer.
3. Fit any learned transform (feature selection, scaling, SVD) on training cells only.
4. Pick the likelihood from the table above; add `recon_targets_spec` for binomial-type data.
5. Size encoders to the input: wide inputs (thousands of features) get wider first layers.
6. Balance modalities with `recon_weight` if one dominates the loss.